# Daily limit-order candidate deep dive

Build a candidate from walk-forward selections, then inspect its full-sample contribution-lot execution behavior.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from retail_sp500.daily_data import daily_data_summary, load_or_fetch_twelve_data_daily

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")

from retail_sp500.limit_portfolio import (
    RecurringLimitConfig,
    evaluate_recurring_limit_grid,
    simulate_recurring_limit_strategy,
    summarize_recurring_limit_result,
    walk_forward_recurring_limit_selection,
)

In [ ]:
SYMBOL = "SPY"
START_DATE = "2007-06-01"
CACHE_PATH = Path("data/processed/spy_daily_1day.csv")
REFRESH = False

daily = load_or_fetch_twelve_data_daily(
    os.getenv("TWELVE_DATA_API_KEY"),
    cache_path=CACHE_PATH,
    refresh=REFRESH,
    symbol=SYMBOL,
    start_date=START_DATE,
)

source = daily_data_summary(daily, symbol=SYMBOL)
print(source)
assert source["interval"] == "1day"
assert daily.index.max() <= pd.Timestamp.today().normalize()
daily.tail()

## Derive a candidate from out-of-sample selections

In [ ]:
DISCOUNTS = np.arange(0.0, 0.0301, 0.001)
MAX_WAIT_SESSIONS = 5

walk_forward = walk_forward_recurring_limit_selection(
    daily,
    DISCOUNTS,
    train_years=5,
    test_years=1,
    max_wait_sessions=MAX_WAIT_SESSIONS,
    monthly_contribution=1_000,
)

CANDIDATE_DISCOUNT = float(walk_forward["selected_discount"].median())
CANDIDATE_DISCOUNT

## Full-sample sensitivity around the candidate

In [ ]:
nearby = DISCOUNTS[
    (DISCOUNTS >= max(0.0, CANDIDATE_DISCOUNT - 0.005))
    & (DISCOUNTS <= CANDIDATE_DISCOUNT + 0.005)
]

sensitivity = evaluate_recurring_limit_grid(
    daily,
    nearby,
    max_wait_sessions=MAX_WAIT_SESSIONS,
    initial_cash=100_000,
    monthly_contribution=1_000,
)

sensitivity.sort_values("ending_excess_value", ascending=False)

## Contribution-lot trace

In [ ]:
lots = simulate_recurring_limit_strategy(
    daily,
    RecurringLimitConfig(
        discount=CANDIDATE_DISCOUNT,
        max_wait_sessions=MAX_WAIT_SESSIONS,
        initial_cash=100_000,
        monthly_contribution=1_000,
    ),
)

summarize_recurring_limit_result(lots)

In [ ]:
lots[[
    "amount",
    "fill_date",
    "fill_type",
    "wait_sessions",
    "baseline_open_price",
    "fill_price",
    "execution_savings_vs_immediate_open",
    "ending_excess_value",
]].tail(24)

In [ ]:
annual_execution = lots.assign(year=lots.index.year).groupby("year").agg(
    lots=("amount", "size"),
    contributed=("amount", "sum"),
    average_wait=("wait_sessions", "mean"),
    forced_fill_rate=("fill_type", lambda values: (values == "expiry_close").mean()),
    average_execution_savings=("execution_savings_vs_immediate_open", "mean"),
    ending_excess_value=("ending_excess_value", "sum"),
)

annual_execution

In [ ]:
px.bar(
    annual_execution.reset_index(),
    x="year",
    y="ending_excess_value",
    title="Candidate execution advantage by contribution year",
).show()

## Remaining requirements before live use

Add dividends, cash yield, spreads, fees, taxes, order cut-off times, and a broker-specific order lifecycle. The candidate is a research result, not an instruction to trade.